# SNC — Evaluate the Conditioned Separator

Runs every test the codebase ships with, against the latest trained
model on Drive. Use this after each training run to check that the
model has not collapsed and that detection / removal still work.

| Stage | Script | What it measures |
|------|--------|------------------|
| 1 | `diagnose_model.py` | Smoke test — is the model alive and class-conditional? |
| 2 | `evaluate_conditioned_separator.py` | SI-SDR / SI-SDRi on synthetic positive-only mixtures |
| 3 | `evaluate_detection.py` | Precision / Recall / F1 on synthetic multi-class mixtures |
| 4 | `test_realworld.py` | Detection + optional removal on a user-uploaded audio file |

Pick the **T4** GPU runtime (newer GPUs can fail with `CUDA_ERROR_INVALID_PTX`).

## 1. Mount Drive and clone the repo

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DRIVE_ROOT = '/content/drive/MyDrive/snc'
GITHUB_USER = 'keremtutumlu'
GITHUB_REPO_NAME = 'selective-noise-cancelling'
BRANCH = 'feature/separator-quality-overhaul'

# Private repo? Add a Colab Secret named GITHUB_TOKEN (key icon, scope: repo).
import os, subprocess, shutil
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

clone_url = (f'https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
             if token else
             f'https://github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git')

REPO_DIR = Path(f'/content/{GITHUB_REPO_NAME}')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
result = subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('git clone failed — if the repo is private add a '
                       'GITHUB_TOKEN secret in the left sidebar.')
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

Working dir: /content/selective-noise-cancelling


In [3]:
# Symlink data and saved_models to Drive so the trained model and datasets
# are accessible without re-downloading.
drive_data = Path(DRIVE_ROOT) / 'data'
drive_models = Path(DRIVE_ROOT) / 'saved_models'

for local, target in [(REPO_DIR / 'data', drive_data),
                       (REPO_DIR / 'saved_models', drive_models)]:
    if local.is_symlink() or local.exists():
        local.unlink() if local.is_symlink() else shutil.rmtree(local)
    local.symlink_to(target)
    print(f'{local} -> {target}')

ckpt = drive_models / 'separation_models' / 'separator_unet_film_multi_v2.0.h5'
assert ckpt.exists(), f'Trained model not found at {ckpt} — run training first.'
print('Model found:', ckpt)

/content/selective-noise-cancelling/data -> /content/drive/MyDrive/snc/data
/content/selective-noise-cancelling/saved_models -> /content/drive/MyDrive/snc/saved_models
Model found: /content/drive/MyDrive/snc/saved_models/separation_models/separator_unet_film_multi_v2.0.h5


In [4]:
!pip install -q librosa==0.11.0 soundfile scikit-learn pandas
import tensorflow as tf
print('TF version :', tf.__version__)
print('GPUs       :', tf.config.list_physical_devices('GPU'))

TF version : 2.20.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Smoke test — is the model alive?

Checks three things in under a minute:

1. Model produces non-zero output for known present classes.
2. Different class queries on the same input produce **different** outputs (FiLM is doing something).
3. Across many classes, the correct query consistently beats wrong queries.

If this fails, the model has collapsed and there is no point running the
longer evaluations — retrain with safer hyperparameters first.

In [5]:
!python src/model_training/diagnose_model.py


Model diagnostic — separator_unet_film_multi_v2.1.h5
Data root    — /content/selective-noise-cancelling/data/raw

Traceback (most recent call last):
  File "/content/selective-noise-cancelling/src/model_training/diagnose_model.py", line 154, in <module>
    diagnose(
  File "/content/selective-noise-cancelling/src/model_training/diagnose_model.py", line 70, in diagnose
    model = tf.keras.models.load_model(model_path, compile=False)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_api.py", line 196, in load_model
    return legacy_h5_format.load_model_from_hdf5(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras/src/legacy/saving/legacy_h5_format.py", line 118, in load_model_from_hdf5
    f = h5py.File(filepath, mode="r")
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/h5py/_hl/files.py", line 555, in __ini

## 3. SI-SDR — separation quality on synthetic mixtures

Builds 800 positive-only test mixtures (the query class is always
present), runs separation, and scores SI-SDR against the ground-truth
stem. The headline metric is **SI-SDRi** — improvement over the
unprocessed mixture.

Healthy ranges:
- **+5 dB or more** — working
- **0–2 dB** — under-trained
- **≤ 0 dB** — model is making it worse (collapsed)

In [6]:
!python src/model_training/evaluate_conditioned_separator.py


Conditioned Separator — Evaluation Report
Model : /content/selective-noise-cancelling/saved_models/separation_models/separator_unet_film_multi_v2.1.h5

Traceback (most recent call last):
  File "/content/selective-noise-cancelling/src/model_training/evaluate_conditioned_separator.py", line 154, in <module>
    ).evaluate()
      ^^^^^^^^^^
  File "/content/selective-noise-cancelling/src/model_training/evaluate_conditioned_separator.py", line 95, in evaluate
    mixer = SeparationMixer(self.data_root, negative_prob=0.0, seed=self.seed)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/selective-noise-cancelling/src/data_preparation/separation_mixer.py", line 79, in __init__
    self._waveform_cache: Dict[str, List[np.ndarray]] = load_all_datasets(
                                                        ^^^^^^^^^^^^^^^^^^
  File "/content/selective-noise-cancelling/src/data_preparation/dataset_sources.py", line 84, in load_all_datasets
    f

## 4. Detection — precision / recall on synthetic multi-class mixtures

Builds 200 mixtures of 1–4 random classes, runs the webapp detection
scoring, and reports per-class precision, recall, F1. This tests the
detect_sounds path independently from separation.

Look for mean F1 of at least **0.5** — below that the surfaced classes
are largely noise.

In [7]:
!python src/model_training/evaluate_detection.py


Detection Evaluation — separator_unet_film_multi_v2.1.h5
Test mixtures: 200

Traceback (most recent call last):
  File "/content/selective-noise-cancelling/src/model_training/evaluate_detection.py", line 139, in <module>
    evaluate(
  File "/content/selective-noise-cancelling/src/model_training/evaluate_detection.py", line 81, in evaluate
    model = tf.keras.models.load_model(model_path, compile=False)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_api.py", line 196, in load_model
    return legacy_h5_format.load_model_from_hdf5(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras/src/legacy/saving/legacy_h5_format.py", line 118, in load_model_from_hdf5
    f = h5py.File(filepath, mode="r")
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/h5py/_hl/files.py", line 555, in __init__
    fid = make_fid(name, 

## 5. Real-world audio — your own file

Upload a WAV / MP3 / MP4 file, then run the test. It prints the full
detection scoreboard (top 15 with scores) and optionally separates a
list of classes you specify, writing `<filename>_cleaned.wav` next to
the input.

In [ ]:
from google.colab import files
uploaded = files.upload()
audio_path = Path(list(uploaded.keys())[0])
print('Uploaded:', audio_path)

In [ ]:
# Detection only (no separation). To remove classes, append them as args:
#   !python src/model_training/test_realworld.py {audio_path} siren dog
# Optional: --strength 0.7
!python src/model_training/test_realworld.py {audio_path}

In [ ]:
# Example: remove the strongest detected class at full strength.
# Replace 'siren' with whatever the detection above surfaced.
# !python src/model_training/test_realworld.py {audio_path} siren --strength 1.0